In [50]:
# Cellule 1 : Imports et configuration
%load_ext autoreload
%autoreload 2

import pandas as pd

# Import des classe et fonctions personnalisées
from tools.OI_class_v2 import OI_DataProcessor
from tools.OI_utils import extract_xls,analyser_correlation_croisee

# Configuration de l'affichage pour voir toutes les colonnes
pd.set_option('display.max_columns', None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [51]:
tags = [
        {'tag':'WQ33222VA', 'nom':'Ester','info':'Totalisation du Peson Acetate/Propionate' },
        {'tag':'3340_type', 'nom':'A/P','info':'Acetate ou Propionate' },
        {'tag':'CTY_ACV43A_Teneur Vit. A (UV)', 'nom':'Acetate_UV','info':'ACV43A teneur en acetate'},
        {'tag':'CTY_A3340FGB_Teneur arr. Vit. A (UV)', 'nom':'Propionate_UV', 'info':'A3340FGB teneur en propionate'}
    ]

In [55]:
# Initialisation
processor = OI_DataProcessor(
    url_base = 'https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    tags_other = [],
    tags_selected = tags,
    start='2026-05-12',
    end='2026-05-14',
    interval='PT1H',
    verbose=False
)

# Charger les données
processor.merge()

# Appliquer des filtres (chaînage possible)
#processor.filtering(
#    tag=['A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Labo','A1000R_Poids', 'A1000R_UV_Auto', 'A1000R_Labo','Acetate_UV'],
#    min_val=[700, 800_000, 700_000, 700, 800_000, 700_000, 1_500_000],
#    max_val=[1500, 1_300_000, 1_300_000, 1500, 1_300_000, 1_300_000, 10_000_000],
#    na=['A1000M_Poids','A1000R_Poids']
#)

# Ajouter des colonnes
processor.ajoute_calcul('Ester','A/P','Acetate_UV','Propionate_UV',1e-9,'cumul')
#processor.ajoute_cumul('A1000M_Poids', 'A1000M_UV_Auto', 1000, 'Cumul_UV')
#processor.ajouter_moyennes_glissantes_ponderee('A1000M_Poids', 'A1000M_UV_Auto', 'A1000M_Moyenne_UV', window=15)
#processor.ajouter_moyennes_glissantes('Acetate_UV', 'Acetate_Moyenne_UV', window=15)

# Afficher les infos finales
processor.info()

INFORMATIONS DataProcessor
URL de base       : https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?
Tags autres       : []
Tags sélectionnés : [{'tag': 'WQ33222VA', 'nom': 'Ester', 'info': 'Totalisation du Peson Acetate/Propionate'}, {'tag': '3340_type', 'nom': 'A/P', 'info': 'Acetate ou Propionate'}, {'tag': 'CTY_ACV43A_Teneur Vit. A (UV)', 'nom': 'Acetate_UV', 'info': 'ACV43A teneur en acetate'}, {'tag': 'CTY_A3340FGB_Teneur arr. Vit. A (UV)', 'nom': 'Propionate_UV', 'info': 'A3340FGB teneur en propionate'}]
Tous les tags API : ['WQ33222VA', '3340_type', 'CTY_ACV43A_Teneur Vit. A (UV)', 'CTY_A3340FGB_Teneur arr. Vit. A (UV)']
Mapping renommage : {'WQ33222VA': 'Ester', '3340_type': 'A/P', 'CTY_ACV43A_Teneur Vit. A (UV)': 'Acetate_UV', 'CTY_A3340FGB_Teneur arr. Vit. A (UV)': 'Propionate_UV'}
Date de début     : 2026-05-12
Date de fin       : 2026-05-14
Intervalle        : PT1H
Heure de début    : 00
Heure de fin      : 23
--------------------------------------------

In [56]:
# 1. On veut changer de date ?
processor.start = '2026-05-01'
processor.end = '2026-05-26'

# 2. On recalcule TOUT (merge + filtres + cumul) en une commande
processor.recalculate()

# 3. On vérifie le pipeline
processor.show_pipeline()

processor.data.describe()


--- PIPELINE ACTUEL ---
1. ajoute_calcul | Args: ('Ester', 'A/P', 'Acetate_UV', 'Propionate_UV', 1e-09, 'cumul') | Kwargs: {}
-----------------------



,Ester,A/P,Acetate_UV,Propionate_UV,cumul
count,6.240000e+02,624.000000,6.240000e+02,6.240000e+02,624.000000
mean,2.644724e+06,0.126489,2.252419e+06,2.296625e+06,114.618484
std,2.891822e+04,0.331604,2.395962e+04,4.268458e+04,65.406260
min,2.593880e+06,0.000000,2.204082e+06,2.188000e+06,0.000000
25%,2.620030e+06,0.000000,2.234690e+06,2.314000e+06,58.516525
50%,2.644910e+06,0.000000,2.252979e+06,2.314000e+06,115.095991
75%,2.668826e+06,0.000000,2.264770e+06,2.314000e+06,168.738169
max,2.694550e+06,1.000000,2.293140e+06,2.318000e+06,227.561403


In [54]:
nb = processor.data['Ester'].size
ester = (processor.data['Ester'][nb-1] - processor.data['Ester'][0]) * processor.data['Acetate_UV'][10] * 1e-9
print(nb, ester, processor.data['Ester'])

624 224.58107888 timestamp
2026-05-01 00:00:00+00:00    2.593880e+06
2026-05-01 01:00:00+00:00    2.593880e+06
2026-05-01 02:00:00+00:00    2.593880e+06
2026-05-01 03:00:00+00:00    2.593880e+06
2026-05-01 04:00:00+00:00    2.594335e+06
                                 ...     
2026-05-26 19:00:00+00:00    2.693481e+06
2026-05-26 20:00:00+00:00    2.694550e+06
2026-05-26 21:00:00+00:00    2.694550e+06
2026-05-26 22:00:00+00:00    2.694550e+06
2026-05-26 23:00:00+00:00    2.694550e+06
Name: Ester, Length: 624, dtype: float64
